# Data Chunking Pipeline for Palantir Foundry Envision

This notebook reads chunking instructions from a text file and generates the necessary chunked and ontology-ready files for use in Foundry Envision and Ontology Management_Server.

## 1. Read Instructions from File

Read the contents of `chunking_instructions.txt` to understand the chunking and file generation requirements.

In [26]:
# Read the chunking instructions
with open('../data_processing/chunked_data/chunking_instructions.txt', 'r') as f:
    instructions = f.read()
print(instructions)

Title: data_chunking_pipeline.ipynb
Location: ../../code

Please generate a Jupyter notebook named data_chunking_pipeline.ipynb inside the ../../code folder.

🎯 Purpose

I want to preprocess multiple CSV files so I can pass the data to an LLM inside Palantir Foundry Envision without hitting token limits.

Raw tables contain multiple long-text fields, so they must be chunked.

I will use:

Foundry Ontology Objects for structured navigation

Chunked text + Embeddings for semantic retrieval

The output must be formatted to work with Envision Pipelines, Ontology Manager, and LLM Agents.

Requirements (Best Practice for Multiple Tables)

1. Load & Process All CSV Files Together

Load all CSVs from:

/Users/mathewthomas/Documents/hobby_projects/AI_ML_Work/567COG_DS_AI_ML/Usecases/A9A_Assessment/data

2. Identify and Chunk ALL Long-Text Columns

For each CSV:

-Detect columns containing more than 25 words

-These columns must be chunked independently

-But output must maintain one combined ro

## 2. Parse Chunking Instructions

Extract the requirements for chunking and file generation from the instructions.

In [23]:
import re

# Example: parse for chunking and file generation requirements
def parse_instructions(text):
    requirements = {}
    # Use correct absolute path for the data folder and chunked output
    requirements['chunked_folder'] = '/Users/mathewthomas/Documents/hobby_projects/AI_ML_Work/567COG_DS_AI_ML/Usecases/A9A_Assessment/data_processing/chunked_data'
    requirements['data_folder'] = '/Users/mathewthomas/Documents/hobby_projects/AI_ML_Work/567COG_DS_AI_ML/Usecases/A9A_Assessment/data'
    requirements['chunk_suffix'] = '_was_chunked'
    requirements['no_chunk_suffix'] = '_not_chunked'
    # Add more parsing logic as needed
    return requirements

requirements = parse_instructions(instructions)
print(requirements)

{'chunked_folder': '/Users/mathewthomas/Documents/hobby_projects/AI_ML_Work/567COG_DS_AI_ML/Usecases/A9A_Assessment/data_processing/chunked_data', 'data_folder': '/Users/mathewthomas/Documents/hobby_projects/AI_ML_Work/567COG_DS_AI_ML/Usecases/A9A_Assessment/data', 'chunk_suffix': '_was_chunked', 'no_chunk_suffix': '_not_chunked'}


## 3. Generate Necessary Files Based on Instructions

Process each CSV in the data folder, chunk long text fields, and save the results in the `chunked_data` folder with the appropriate suffix. Also, prepare data for Ontology Management_Server.

In [30]:
import pandas as pd
import os
from pathlib import Path
import uuid

def detect_long_text_columns(df, min_words=25):
    return [col for col in df.columns if df[col].dtype == object and df[col].astype(str).apply(lambda x: len(x.split())).max() > min_words]

def split_into_chunks(text, chunk_size=25, overlap=2):
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunk = words[i:i+chunk_size]
        if not chunk:
            break
        chunks.append(' '.join(chunk))
        if i + len(chunk) >= len(words):
            break
        i += max(1, len(chunk) - overlap)
    return chunks

def process_file(file_path):
    df = pd.read_csv(file_path)
    long_text_cols = detect_long_text_columns(df)
    chunked_rows = []
    for idx, row in df.iterrows():
        row_dict = row.to_dict()
        # Keep all boolean and short string columns as metadata
        metadata_fields = {k: v for k, v in row_dict.items() if (df[k].dtype == bool or (df[k].dtype == object and df[k].astype(str).apply(lambda x: len(x.split())).max() <= 10))}
        # Always keep Mission_Title or mission_title if present
        for mcol in ['Mission_Title', 'mission_title']:
            if mcol in row_dict:
                metadata_fields[mcol] = row_dict[mcol]
        if long_text_cols:
            for col in long_text_cols:
                text = str(row[col])
                if len(text.split()) > 25:
                    chunks = split_into_chunks(text, chunk_size=25, overlap=2)
                    for chunk_idx, chunk in enumerate(chunks):
                        chunk_row = metadata_fields.copy()
                        chunk_row['text_chunk'] = chunk
                        chunk_row['source_file'] = Path(file_path).name
                        chunk_row['source_column'] = col
                        chunk_row['chunk_index'] = chunk_idx + 1
                        chunk_row['column_chunked'] = col
                        chunk_row['chunk_id'] = str(uuid.uuid4())
                        chunk_row['original_row_id'] = idx
                        chunked_rows.append(chunk_row)
                else:
                    chunk_row = metadata_fields.copy()
                    chunk_row['text_chunk'] = text
                    chunk_row['source_file'] = Path(file_path).name
                    chunk_row['source_column'] = col
                    chunk_row['chunk_index'] = 1
                    chunk_row['column_chunked'] = col
                    chunk_row['chunk_id'] = str(uuid.uuid4())
                    chunk_row['original_row_id'] = idx
                    chunked_rows.append(chunk_row)
        else:
            chunk_row = metadata_fields.copy()
            chunk_row['text_chunk'] = ''
            chunk_row['source_file'] = Path(file_path).name
            chunk_row['source_column'] = None
            chunk_row['chunk_index'] = 1
            chunk_row['column_chunked'] = None
            chunk_row['chunk_id'] = str(uuid.uuid4())
            chunk_row['original_row_id'] = idx
            chunked_rows.append(chunk_row)
    was_chunked = bool(long_text_cols)
    return pd.DataFrame(chunked_rows), was_chunked, long_text_cols

def save_output(df, was_chunked, filename):
    suffix = '_was_chunked' if was_chunked else '_not_chunked'
    out_path = chunked_dir / (filename + suffix + '.csv')
    df.to_csv(out_path, index=False)
    return out_path

chunked_dir = Path('/Users/mathewthomas/Documents/hobby_projects/AI_ML_Work/567COG_DS_AI_ML/Usecases/A9A_Assessment/data_processing/chunked_data')
chunked_dir.mkdir(exist_ok=True)
data_dir = Path('/Users/mathewthomas/Documents/hobby_projects/AI_ML_Work/567COG_DS_AI_ML/Usecases/A9A_Assessment/data')

log_summary = []
all_chunked_paths = []

for csv_file in data_dir.glob('*.csv'):
    print(f"Processing file: {csv_file.name}")
    chunked_df, was_chunked, long_text_cols = process_file(csv_file)
    out_path = save_output(chunked_df, was_chunked, csv_file.stem)
    if was_chunked:
        all_chunked_paths.append(out_path)
    log_summary.append({
        'file': csv_file.name,
        'rows': len(chunked_df),
        'long_text_columns': ','.join(long_text_cols),
        'chunks_generated': len(chunked_df)
    })

log_path = chunked_dir / 'chunking_log_summary.csv'
pd.DataFrame(log_summary).to_csv(log_path, index=False)
print(f"Saved log summary: {log_path.resolve()}")

# Section: Create single chunked document for Foundry
single_chunks = []
all_columns = set()
for chunked_path in all_chunked_paths:
    df = pd.read_csv(chunked_path)
    all_columns.update(df.columns)
    single_chunks.append(df)

# Ensure all DataFrames have the same columns (union of all columns)
all_columns = list(all_columns)
for i, df in enumerate(single_chunks):
    missing_cols = set(all_columns) - set(df.columns)
    for col in missing_cols:
        df[col] = ''
    single_chunks[i] = df[all_columns]

single_chunked_doc = pd.concat(single_chunks, ignore_index=True)
single_doc_path = chunked_dir / 'a9a_assessment_single_chunked_document.csv'
single_chunked_doc.to_csv(single_doc_path, index=False)
print(f"Saved single chunked document for Foundry: {single_doc_path.resolve()} (rows: {len(single_chunked_doc)})")

Processing file: Mission_Metrics.csv
Processing file: Equipment_Issues.csv
Processing file: FFIRs_Friction_Points_Recommendations.csv
Processing file: Tactical_Lessons_Learned.csv
Processing file: Missions_by_Location_and_Tasked_System.csv
Processing file: Cyber_Document_Library.csv
Processing file: Mission_Details.csv
Processing file: Hardening_Recommendations.csv
Saved log summary: /Users/mathewthomas/Documents/hobby_projects/AI_ML_Work/567COG_DS_AI_ML/Usecases/A9A_Assessment/data_processing/chunked_data/chunking_log_summary.csv
Saved single chunked document for Foundry: /Users/mathewthomas/Documents/hobby_projects/AI_ML_Work/567COG_DS_AI_ML/Usecases/A9A_Assessment/data_processing/chunked_data/a9a_assessment_single_chunked_document.csv (rows: 630)


## 4. Verify Generated Files

List and display the generated files in the `chunked_data` folder to confirm they match the instructions.

In [25]:
# List generated files in chunked_data folder
from pathlib import Path
chunked_dir = Path(requirements['chunked_folder'])
print('Generated files:')
for f in chunked_dir.glob('*.csv'):
    print(f.name)

Generated files:
Mission_Details_not_chunked.csv
Tactical_Lessons_Learned_was_chunked.csv
Mission_Metrics_not_chunked.csv
Equipment_Issues_was_chunked.csv
Cyber_Document_Library_not_chunked.csv
FFIRs_Friction_Points_Recommendations_was_chunked.csv
Missions_by_Location_and_Tasked_System_not_chunked.csv
Hardening_Recommendations_was_chunked.csv
